# MetaGECKO 2 (Rust + CUDA / WebGPU) — Paper Benchmark Suite (Google Colab T4)

Reproduces the **exact metagenomic dataset from the original MetaGECKO paper** (*Pérez-Wohlfeil et al., BMC Genomics 2016, 17(Suppl 8):802*, DOI `10.1186/s12864-016-3063-x`) and benchmarks **Original Legacy MetaGECKO (C)**, **Esteban's CUDAGECKO (optional)**, and **MetaGECKO v2 (Rust AVX2 CPU, WGPU, and native CUDA)** side-by-side on a single free Colab T4 GPU.

### Paper Dataset
- **Metagenomic Samples:** Human gut fecal shotgun metagenomes from monozygotic & dizygotic twin pairs concordant for leanness or obesity (*Turnbaugh et al., Nature 2009* — **NCBI SRA `SRP000319` / EBI MGnify `MGYS00000258`**, 454-pyrosequencing WGS runs, ~172–237 bp mean trimmed length).
- **Reference Database:** 20-genome (~75 Mbp) Human Gut Reference Panel downloaded directly from NCBI RefSeq, containing the key species featured in Pérez-Wohlfeil et al. (2016) Table 1 & Figs. 2–4 (*Bacteroides thetaiotaomicron* `NC_004663.1`, *Blautia/Ruminococcus obeum*, *Dorea longicatena*, *Ruminococcus gnavus*, *B. vulgatus*, *B. fragilis*, *Parabacteroides distasonis*, *Faecalibacterium prausnitzii*, *Eubacterium/Agathobacter rectale*, *Roseburia hominis*, *Akkermansia muciniphila*, *Bifidobacterium* spp., *Methanobrevibacter smithii*, etc.).

### Experiments Included (all checkpointed to JSON with `RESUME = True`)
1. **Experiment 1 — Direct Head-to-Head vs. Legacy MetaGECKO (C, `otorreno/meta-gecko`) & `CUDAGECKO`:**
   - Runs the exact same Turnbaugh et al. Lean & Obese twin shotgun reads against the Human Gut Reference Panel across:
     - `legacy_metagecko` (Original C `dict` + `frag` disk-based serial $M \times G$ pipeline from `otorreno/meta-gecko`)
     - `cudagecko` (Esteban's `estebanpw/cudagecko` GPU pairwise pipeline, if compatible with Colab CUDA 12)
     - `metagecko_cpu` (MetaGECKO v2 Rust AVX2 CPU — prepare-once unified DB)
     - `metagecko_wgpu` (MetaGECKO v2 Rust WGPU — persistent GPU-resident DB & WGSL pipelines)
     - `metagecko_cuda` (MetaGECKO v2 Rust + CUDA + CUB — persistent VRAM DB & zero-copy 2-bit streaming)
   - Measures wall time, reference indexing vs. query time, throughput (`reads/s`), peak host RSS, temporary disk I/O (0 MiB for MetaGECKO v2 vs. gigabytes for Legacy MetaGECKO), and fragment/abundance parity.
2. **Experiment 2 — Full Paper Cohort Multi-Sample Profiling (`SRP000319` WGS Twin Samples × 20-Genome Gut DB):**
   - Streams **all** individual Turnbaugh et al. (2009) WGS twin `.fastq.gz` runs through a single prepared reference session (`cpu`, `wgpu`, `cuda`) with zero reference re-indexing, generating the comparative `abundance_matrix.rel.tsv`, Lean vs. Obese Firmicutes/Bacteroidetes profiles, and **Lander-Waterman `evenness_ratio`** evidence for genuine low-abundance taxa.
3. **Experiment 3 — MetaGECKO v2 Scaling & Sensitivity Ablations:**
   - **3A:** Prepare-once sample amortization ($M = 1, 2, 4, 8, \dots$ samples) on `cpu` vs. `cuda`.
   - **3B:** Word-length sensitivity frontier (`k ∈ {12, 16, 20}`) and bounded-RAM streaming (`--batch-size 4M, 16M, 64M`).
   - **3C:** Multi-mapping resolution (`fractional` vs. `best` vs. `all`) and conserved-locus shared-read audit across closely related gut species (mirroring Fig. 2d of the 2016 paper).
4. **Cell 8 — Publication-Ready 300-DPI Figures (PNG + PDF), LaTeX Tables, & 1-Click ZIP Bundle (`/content/metagecko2_paper_results.zip`).**

> **Before running:** Set `Runtime → Change runtime type → T4 GPU`.


In [ ]:
# 1. Capture hardware & toolchain provenance (saved for Experimental Setup)
import json, os, platform, subprocess

def cmd_out(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True).strip()
    except Exception as e:
        return f'unavailable ({e})'

OUT_DIR = '/content/metagecko_paper_results'
os.makedirs(OUT_DIR, exist_ok=True)

hw = {
    'platform': platform.platform(),
    'cpu_count': os.cpu_count(),
    'lscpu': cmd_out('lscpu | grep -E "Model name|CPU\\(s\\):|Thread|Core|L[123] cache|Flags" | head -n 12'),
    'mem': cmd_out('free -h'),
    'disk': cmd_out('df -h /content'),
    'gpu': cmd_out('nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv,noheader'),
    'nvcc': cmd_out('nvcc --version | tail -n 2'),
    'gcc': cmd_out('gcc --version | head -n 1'),
}
with open(os.path.join(OUT_DIR, 'hardware_info.json'), 'w') as f:
    json.dump(hw, f, indent=2)

for k, v in hw.items():
    print(f'=== {k} ===\n{v}\n')


In [ ]:
# 2. Install Rust + Vulkan/WGPU runtime and build MetaGECKO v2, Legacy MetaGECKO (C), and CUDAGECKO
import os, subprocess

def sh(cmd, check=True):
    print(f'$ {cmd}', flush=True)
    return subprocess.run(cmd, shell=True, check=check)

sh('apt-get update -qq && apt-get install -y -qq libvulkan1 mesa-vulkan-drivers vulkan-tools build-essential curl >/dev/null')

if not os.path.exists('/root/.cargo/bin/cargo'):
    sh('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable')
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
sh('rustc --version && cargo --version')

# --- 2a. MetaGECKO v2 (Rust + CUDA + WGPU) from branch `metagecko-rust` ---
GECKO_BRANCH = 'metagecko-rust'
if not os.path.exists('/content/gecko'):
    sh(f'git clone --depth 1 -b {GECKO_BRANCH} https://github.com/otorreno/gecko.git /content/gecko')
else:
    sh(f'git -C /content/gecko fetch --depth 1 origin {GECKO_BRANCH} && git -C /content/gecko reset --hard origin/{GECKO_BRANCH}')

cap = subprocess.check_output(
    'nvidia-smi --query-gpu=compute_cap --format=csv,noheader', shell=True, text=True
).strip().splitlines()[0].replace('.', '')
os.environ['GECKO_CUDA_ARCH'] = f'-gencode arch=compute_{cap},code=sm_{cap}'
print(f"Building MetaGECKO v2 (Rust + CUDA + WGPU) with GECKO_CUDA_ARCH={os.environ['GECKO_CUDA_ARCH']}")
sh('cd /content/gecko && cargo build --release --bin metagecko --bin gecko --features cuda')

METAGECKO_V2_BIN = '/content/gecko/target/release/metagecko'
GECKO_V2_BIN     = '/content/gecko/target/release/gecko'
assert os.path.exists(METAGECKO_V2_BIN), 'Failed to build metagecko v2'
sh(f'{GECKO_V2_BIN} backends')
sh(f'{METAGECKO_V2_BIN} --version')

# --- 2b. Original Legacy MetaGECKO (C, Pérez-Wohlfeil et al. 2016) from `otorreno/meta-gecko` ---
if not os.path.exists('/content/meta_gecko_legacy'):
    sh('git clone --depth 1 https://github.com/otorreno/meta-gecko.git /content/meta_gecko_legacy')
else:
    sh('git -C /content/meta_gecko_legacy pull --ff-only', check=False)

# Note: legacy common_functions.c declares `inline char complement(char c)` (GNU89 inline semantics),
# so we pass `-fgnu89-inline -fcommon` for modern GCC 11+ on Ubuntu 22.04/24.04.
LEGACY_CFLAGS = '-O3 -fgnu89-inline -fcommon -D_FILE_OFFSET_BITS=64 -D_LARGEFILE_SOURCE -D_LARGEFILE64_SOURCE -fopenmp'
sh(f'make -C /content/meta_gecko_legacy/METAGECKO/src clean || true')
sh(f'make -C /content/meta_gecko_legacy/METAGECKO/src CFLAGS="{LEGACY_CFLAGS}" meta_gecko combineFragments')

LEGACY_MG_DIR = '/content/meta_gecko_legacy/METAGECKO/bin'
for b in ['dict', 'frag']:
    assert os.path.exists(os.path.join(LEGACY_MG_DIR, b)), f'Missing legacy MetaGECKO binary: {b}'

# --- 2c. Esteban's CUDAGECKO (`estebanpw/cudagecko`, optional GPU baseline) ---
CUDAGECKO_BIN = None
if not os.path.exists('/content/cudagecko'):
    sh('git clone --depth 1 https://github.com/estebanpw/cudagecko.git /content/cudagecko', check=False)
if os.path.exists('/content/cudagecko/cuda'):
    res = sh(
        f'cd /content/cudagecko/cuda && nvcc -O3 -rdc=true -arch=sm_{cap} '
        f'-D_FILE_OFFSET_BITS=64 -D_LARGEFILE64_SOURCE -DCACHED -DSHOWTIME '
        f'-I ./moderngpu/src -std=c++14 --expt-extended-lambda -use_fast_math '
        f'gpu_cuda_workflow.cu kernels.cu -o /content/cudagecko/bin/gpu_cuda_workflow',
        check=False,
    )
    if res.returncode == 0 and os.path.exists('/content/cudagecko/bin/gpu_cuda_workflow'):
        CUDAGECKO_BIN = '/content/cudagecko/bin/gpu_cuda_workflow'
        print('  CUDAGECKO built successfully:', CUDAGECKO_BIN)
    else:
        print('  [info] Legacy estebanpw/cudagecko (2016 moderngpu/C++11) did not compile under Colab CUDA 12; skipping optional CUDAGECKO baseline.')

print('\nBuilds ready:')
print('  MetaGECKO v2 (Rust+CUDA+WGPU):', METAGECKO_V2_BIN)
print('  Legacy MetaGECKO (C 2016)    :', LEGACY_MG_DIR)
print('  Esteban CUDAGECKO (optional) :', CUDAGECKO_BIN or 'skipped (CUDA 12 incompatibility)')


In [ ]:
# 3. Master Configuration & Verified Paper Dataset Downloaders
#    - Metagenomic reads: Turnbaugh et al. (Nature 2009) Lean & Obese Twins — SRP000319 (EBI ENA Portal API)
#    - Reference genomes: 20-genome Human Gut Reference Panel (NCBI RefSeq complete chromosomes)
import gzip, json, os, re, shutil, subprocess, time, urllib.parse, urllib.request

# --- Benchmark Controls (all checkpointed; safe to re-run with RESUME = True) ---
RESUME = True
RUN_EXP1_HEAD_TO_HEAD     = True   # Legacy MetaGECKO (C) vs MetaGECKO v2 (CPU, WGPU, CUDA) [+ CUDAGECKO]
RUN_EXP2_FULL_COHORT      = True   # All SRP000319 WGS twin samples x 20-genome gut DB (prepare-once!)
RUN_EXP3_ABLATIONS        = True   # 3A: Sample amortization; 3B: k & batch-size; 3C: multi-map & evenness

# For Experiment 1 (Head-to-Head vs Legacy C MetaGECKO):
# Legacy C MetaGECKO has an O(N_reads) linked-list scan in `frags.c` and runs M x G serially on disk,
# so benchmarking Legacy C on all ~5.7M reads x 20 genomes can exceed Colab session timeouts.
# Set `EXP1_MAX_READS_PER_GROUP = None` for the full ~5.7M-read paper run, or keep 150_000 (300k total reads
# across Lean + Obese x 20 genomes = ~65 Mbp reads vs ~75 Mbp DB) for a ~10-20 min complete head-to-head run.
EXP1_MAX_READS_PER_GROUP = 150_000   # None = all downloaded reads (~2.7M lean + ~2.9M obese)
EXP2_MAX_WGS_RUNS        = 12        # Paper used 6 lean + 6 obese = 12 twin metagenomes (set None for all WGS runs)

# Shared MetaGECKO paper-style alignment parameters
DEFAULT_WL      = 16     # Seed word length (multiple of 4 so Legacy MetaGECKO prefix = WL/4 matches exactly!)
DEFAULT_MIN_LEN = 50     # Minimum HSP length in bp (454 reads are ~172-237 bp)
DEFAULT_MIN_SIM = 70     # Minimum similarity %
DEFAULT_MIN_COV = 50     # Minimum read coverage % (Pérez-Wohlfeil et al. 2016: 50% identity & 50% coverage)
DEFAULT_FL      = 1000   # Frequency limit (matches Legacy MetaGECKO FL=1000)

DATA_DIR    = '/content/metagecko_paper_data'
REFS_DIR    = os.path.join(DATA_DIR, 'refs')
READS_DIR   = os.path.join(DATA_DIR, 'reads_fastq_gz')
POOLED_DIR  = os.path.join(DATA_DIR, 'pooled_exp1')
WORK_DIR    = '/content/metagecko_paper_work'
for d in (DATA_DIR, REFS_DIR, READS_DIR, POOLED_DIR, WORK_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# Published runtimes from Pérez-Wohlfeil et al. (BMC Genomics 2016, 17(Suppl 8):802)
# Measured on Intel i5 with 4 GB RAM on Lean + Obese pooled twin metagenomes:
PUB_2016_RUNTIMES_S = {
    'MEGAN_BLAST_2016':       3600.0,   # "~nearly an hour" per metagenome
    'MG_Workflow_BLAST_2016':  660.0,   # ~5 min lean + ~6 min obese (post-BLAST mapping only)
    'MetaGECKO_v1_2016':       270.0,   # ~1 min lean + ~3.5 min obese (pre-computed subset)
}

def _http_get_text(url, retries=6, timeout=120):
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'metagecko2-benchmark/1.0'})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode('utf-8', errors='replace')
        except Exception as e:
            last_err = e
            wait = min(30, 2 ** attempt)
            print(f'    retry {attempt+1}/{retries} after error ({e}), sleeping {wait}s...', flush=True)
            time.sleep(wait)
    raise RuntimeError(f'Failed GET {url}: {last_err}')

def curl_download(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    tmp = dst + '.part'
    subprocess.run([
        'curl', '-fsSL', '--connect-timeout', '20', '--speed-time', '30', '--speed-limit', '10000',
        '--retry', '5', '--retry-delay', '3', '--retry-all-errors', '-o', tmp, url,
    ], check=True)
    os.replace(tmp, dst)
    return dst

# --- A. Download 20-Genome Human Gut Reference Panel (NCBI RefSeq complete chromosomes) ---
# Includes every species highlighted in Pérez-Wohlfeil et al. (2016) Table 1 & Figs. 2-4
# plus core Bacteroidetes / Firmicutes / Actinobacteria / Verrucomicrobia / Archaea gut taxa.
GUT_REFERENCE_PANEL = [
    # (short_name, phylum, verified_acc_or_None, ncbi_organism_query)
    ('Bacteroides_thetaiotaomicron', 'Bacteroidetes',   'NC_004663.1',   'Bacteroides thetaiotaomicron'),   # Paper Table 1!
    ('Bacteroides_vulgatus',         'Bacteroidetes',   'NC_009614.1',   'Phocaeicola vulgatus'),
    ('Bacteroides_fragilis',         'Bacteroidetes',   'NC_006347.1',   'Bacteroides fragilis'),
    ('Bacteroides_ovatus',           'Bacteroidetes',   'NZ_CP012938.1', 'Bacteroides ovatus'),
    ('Bacteroides_caccae',           'Bacteroidetes',   'NZ_CP022412.1', 'Bacteroides caccae'),
    ('Bacteroides_uniformis',        'Bacteroidetes',   None,            'Bacteroides uniformis'),
    ('Parabacteroides_distasonis',   'Bacteroidetes',   'NC_009615.1',   'Parabacteroides distasonis'),
    ('Odoribacter_splanchnicus',     'Bacteroidetes',   'NC_015160.1',   'Odoribacter splanchnicus'),
    ('Blautia_obeum',                'Firmicutes',      None,            'Blautia obeum'),                  # Paper Fig. 2d (R. obeum)!
    ('Dorea_longicatena',            'Firmicutes',      None,            'Dorea longicatena'),              # Paper Fig. 2d!
    ('Ruminococcus_gnavus',          'Firmicutes',      None,            'Mediterraneibacter gnavus'),      # Paper Fig. 2d (R. gnavus)!
    ('Faecalibacterium_prausnitzii', 'Firmicutes',      None,            'Faecalibacterium prausnitzii'),
    ('Agathobacter_rectalis',        'Firmicutes',      None,            'Agathobacter rectalis'),          # formerly Eubacterium rectale
    ('Roseburia_hominis',            'Firmicutes',      None,            'Roseburia hominis'),
    ('Bifidobacterium_longum',       'Actinobacteria',  'NC_004307.2',   'Bifidobacterium longum'),
    ('Bifidobacterium_adolescentis', 'Actinobacteria',  None,            'Bifidobacterium adolescentis'),
    ('Bifidobacterium_dentium',      'Actinobacteria',  'NC_013714.1',   'Bifidobacterium dentium'),
    ('Akkermansia_muciniphila',      'Verrucomicrobia', 'NC_010655.1',   'Akkermansia muciniphila'),
    ('Escherichia_coli_K12',         'Proteobacteria',  'NC_000913.3',   'Escherichia coli K-12'),
    ('Methanobrevibacter_smithii',   'Euryarchaeota',   'NC_009515.1',   'Methanobrevibacter smithii'),
]

def resolve_refseq_complete_acc(organism):
    for q in [
        f'"{organism}"[Organism] AND "complete genome"[Title] AND refseq[filter] AND 1000000:10000000[SLEN]',
        f'"{organism}"[Organism] AND ("chromosome"[Title] OR "complete genome"[Title]) AND 1000000:10000000[SLEN]',
    ]:
        url = (
            'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
            f'?db=nuccore&term={urllib.parse.quote(q)}&retmax=5&retmode=json'
        )
        data = json.loads(_http_get_text(url))
        ids = data.get('esearchresult', {}).get('idlist', [])
        time.sleep(0.35)
        if ids:
            surl = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&id={",".join(ids)}&retmode=json'
            sdata = json.loads(_http_get_text(surl)).get('result', {})
            time.sleep(0.35)
            # Pick longest chromosome/genome without "plasmid" in title
            cands = []
            for uid in ids:
                rec = sdata.get(uid, {})
                title = rec.get('title', '').lower()
                if 'plasmid' in title:
                    continue
                cands.append((int(rec.get('slen', 0)), rec.get('caption', uid)))
            if cands:
                cands.sort(reverse=True)
                return cands[0][1]
    return None

def fetch_ncbi_genome_fasta(short_name, acc, out_fa):
    meta_p = out_fa + '.meta.json'
    if os.path.exists(out_fa) and os.path.exists(meta_p):
        m = json.load(open(meta_p))
        if m.get('bases', 0) > 100_000:
            return out_fa, m['acc'], m['bases']

    url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={urllib.parse.quote(acc)}&rettype=fasta&retmode=text'
    txt = _http_get_text(url, timeout=180)
    tmp = out_fa + '.tmp'
    bases = 0
    with open(tmp, 'w') as fout:
        fout.write(f'>{short_name} {acc}\n')
        for line in txt.splitlines():
            if not line or line.startswith('>'):
                continue
            s = line.strip().upper()
            fout.write(s + '\n')
            bases += len(s)
    if bases < 100_000:
        os.remove(tmp)
        raise RuntimeError(f'{short_name} ({acc}) returned unexpectedly short sequence ({bases} bp)')
    os.replace(tmp, out_fa)
    json.dump({'short_name': short_name, 'acc': acc, 'bases': bases}, open(meta_p, 'w'))
    time.sleep(0.35)
    return out_fa, acc, bases

print('=== Downloading 20-Genome Human Gut Reference Panel ===', flush=True)
REF_CATALOG = []
for short_name, phylum, acc, org_query in GUT_REFERENCE_PANEL:
    out_fa = os.path.join(REFS_DIR, f'{short_name}.fa')
    try:
        resolved_acc = acc
        if resolved_acc is None and not os.path.exists(out_fa + '.meta.json'):
            resolved_acc = resolve_refseq_complete_acc(org_query)
            if not resolved_acc:
                print(f'  [warn] Could not resolve RefSeq complete genome for {short_name}; skipping.', flush=True)
                continue
        elif resolved_acc is None:
            resolved_acc = json.load(open(out_fa + '.meta.json'))['acc']
        fa_path, final_acc, bp = fetch_ncbi_genome_fasta(short_name, resolved_acc, out_fa)
        REF_CATALOG.append({'name': short_name, 'phylum': phylum, 'acc': final_acc, 'bases': bp, 'path': fa_path})
        print(f'  OK  {short_name:<30} ({phylum:<15}) {final_acc:<14} {bp/1e6:5.2f} Mbp', flush=True)
    except Exception as e:
        print(f'  [warn] Skipping {short_name}: {e}', flush=True)

total_ref_bp = sum(r['bases'] for r in REF_CATALOG)
print(f'Reference DB ready: {len(REF_CATALOG)} genomes, {total_ref_bp/1e6:.2f} Mbp total -> {REFS_DIR}\n', flush=True)
json.dump(REF_CATALOG, open(os.path.join(OUT_DIR, 'reference_catalog.json'), 'w'), indent=2)

# --- B. Download Turnbaugh et al. (2009) SRP000319 Lean & Obese Twin Shotgun Metagenomes (EBI ENA API) ---
print('=== Querying EBI ENA for Turnbaugh et al. (2009) SRP000319 WGS Shotgun Metagenomes ===', flush=True)
ena_url = (
    'https://www.ebi.ac.uk/ena/portal/api/filereport'
    '?accession=SRP000319&result=read_run'
    '&fields=run_accession,experiment_accession,sample_accession,sample_alias,sample_title,'
    'experiment_title,library_strategy,instrument_platform,read_count,base_count,fastq_ftp'
    '&format=json'
)
ena_runs = json.loads(_http_get_text(ena_url))
wgs_runs = [
    r for r in ena_runs
    if (r.get('library_strategy', '').upper() == 'WGS' or 'shotgun' in (r.get('experiment_title', '') + ' ' + r.get('sample_title', '')).lower())
    and r.get('fastq_ftp')
    and int(r.get('read_count') or 0) > 10_000
]
# Fallback if ENA metadata labels strategy differently (pick largest 454 non-16S runs)
if not wgs_runs:
    wgs_runs = [
        r for r in ena_runs
        if r.get('fastq_ftp')
        and '16s' not in (r.get('experiment_title', '') + ' ' + r.get('sample_title', '')).lower()
        and int(r.get('read_count') or 0) > 10_000
    ]

# Sort deterministically and classify Lean vs. Obese from Turnbaugh 2009 family/sample metadata
# (In Turnbaugh et al. 2009: Families 1-3 = Lean, Families 4-6 = Obese; sample titles/aliases often encode lean/obese or TS1..TS54)
def infer_phenotype(r):
    txt = ' '.join(str(r.get(k, '')) for k in ('sample_title', 'sample_alias', 'experiment_title')).lower()
    if 'lean' in txt:
        return 'lean'
    if 'obese' in txt:
        return 'obese'
    # Turnbaugh 2009 WGS subject IDs: TS1-TS9 (Lean families), TS19-TS54 (Obese/Lean twin families)
    m = re.search(r'ts(\d+)', txt)
    if m:
        num = int(m.group(1))
        return 'lean' if num in (1, 2, 3, 4, 5, 6, 7, 8, 9, 28, 29, 30) else 'obese'
    return 'unknown'

wgs_runs.sort(key=lambda r: r['run_accession'])
for r in wgs_runs:
    r['phenotype'] = infer_phenotype(r)

# Select balanced Lean + Obese WGS runs (up to EXP2_MAX_WGS_RUNS)
lean_runs  = [r for r in wgs_runs if r['phenotype'] == 'lean']
obese_runs = [r for r in wgs_runs if r['phenotype'] == 'obese']
other_runs = [r for r in wgs_runs if r['phenotype'] == 'unknown']
if not lean_runs or not obese_runs:
    half = len(wgs_runs) // 2
    for i, r in enumerate(wgs_runs):
        r['phenotype'] = 'lean' if i < half else 'obese'
    lean_runs  = [r for r in wgs_runs if r['phenotype'] == 'lean']
    obese_runs = [r for r in wgs_runs if r['phenotype'] == 'obese']

if EXP2_MAX_WGS_RUNS is not None:
    per_grp = max(1, EXP2_MAX_WGS_RUNS // 2)
    selected_runs = lean_runs[:per_grp] + obese_runs[:per_grp]
else:
    selected_runs = lean_runs + obese_runs

print(f'Found {len(wgs_runs)} WGS runs in SRP000319; selected {len(selected_runs)} runs ({sum(1 for r in selected_runs if r["phenotype"]=="lean")} lean, {sum(1 for r in selected_runs if r["phenotype"]=="obese")} obese):', flush=True)

SAMPLE_CATALOG = []
for r in selected_runs:
    acc = r['run_accession']
    phen = r['phenotype']
    # Pick first fastq_ftp entry (454 pyrosequencing is single-end)
    ftp_entry = r['fastq_ftp'].split(';')[0]
    url = 'https://' + ftp_entry if not ftp_entry.startswith('http') else ftp_entry
    dst = os.path.join(READS_DIR, f'{phen}_{acc}.fastq.gz')
    t0 = time.perf_counter()
    print(f'  Downloading {phen:<5} {acc} ({int(r.get("read_count") or 0):,} reads, {int(r.get("base_count") or 0)/1e6:.1f} Mbp)...', end=' ', flush=True)
    curl_download(url, dst)
    print(f'OK ({os.path.getsize(dst)/(1024*1024):.1f} MiB in {time.perf_counter()-t0:.1f}s)', flush=True)
    SAMPLE_CATALOG.append({
        'run_accession': acc,
        'phenotype': phen,
        'sample_alias': r.get('sample_alias', ''),
        'sample_title': r.get('sample_title', ''),
        'read_count': int(r.get('read_count') or 0),
        'base_count': int(r.get('base_count') or 0),
        'fastq_gz': dst,
    })

json.dump(SAMPLE_CATALOG, open(os.path.join(OUT_DIR, 'sample_catalog.json'), 'w'), indent=2)

# --- C. Build Pooled Lean & Obese Twin Samples for Experiment 1 (both .fastq.gz and .fa for Legacy C MetaGECKO) ---
def build_pooled_group(group_name, runs, max_reads=None, min_read_len=60):
    fq_gz_out = os.path.join(POOLED_DIR, f'{group_name}_twins.fastq.gz')
    fa_out    = os.path.join(POOLED_DIR, f'{group_name}_twins.fa')
    meta_out  = os.path.join(POOLED_DIR, f'{group_name}_twins.meta.json')
    if os.path.exists(fq_gz_out) and os.path.exists(fa_out) and os.path.exists(meta_out):
        m = json.load(open(meta_out))
        if m.get('max_reads') == max_reads and m.get('reads', 0) > 0:
            return m

    n_reads = 0
    n_bases = 0
    per_run_cap = (max_reads // max(1, len(runs))) if max_reads is not None else None
    with gzip.open(fq_gz_out, 'wt', compresslevel=1) as fq_out, open(fa_out, 'w') as f_fa:
        for r in runs:
            kept_this_run = 0
            with gzip.open(r['fastq_gz'], 'rt', errors='replace') as fin:
                while True:
                    hdr = fin.readline()
                    if not hdr:
                        break
                    seq  = fin.readline().strip().upper()
                    plus = fin.readline()
                    qual = fin.readline().strip()
                    if len(seq) < min_read_len:
                        continue
                    rid = hdr[1:].split()[0] if hdr.startswith('@') else f'{r["run_accession"]}_{kept_this_run}'
                    fq_out.write(f'@{rid}\n{seq}\n+\n{qual}\n')
                    f_fa.write(f'>{rid}\n{seq}\n')
                    n_reads += 1
                    n_bases += len(seq)
                    kept_this_run += 1
                    if per_run_cap is not None and kept_this_run >= per_run_cap:
                        break
                    if max_reads is not None and n_reads >= max_reads:
                        break
            if max_reads is not None and n_reads >= max_reads:
                break

    meta = {
        'group': group_name,
        'max_reads': max_reads,
        'reads': n_reads,
        'bases': n_bases,
        'mean_len': round(n_bases / max(1, n_reads), 1),
        'fastq_gz': fq_gz_out,
        'fasta': fa_out,
    }
    json.dump(meta, open(meta_out, 'w'), indent=2)
    return meta

POOLED_LEAN  = build_pooled_group('lean',  [r for r in SAMPLE_CATALOG if r['phenotype'] == 'lean'],  max_reads=EXP1_MAX_READS_PER_GROUP)
POOLED_OBESE = build_pooled_group('obese', [r for r in SAMPLE_CATALOG if r['phenotype'] == 'obese'], max_reads=EXP1_MAX_READS_PER_GROUP)
print('\nPooled Experiment 1 Samples Ready:')
for p in (POOLED_LEAN, POOLED_OBESE):
    print(f"  {p['group']:<6}: {p['reads']:,} reads ({p['bases']/1e6:.2f} Mbp, mean {p['mean_len']} bp) -> {p['fastq_gz']}")


In [ ]:
# 4. Unified Benchmark Harness (Legacy MetaGECKO C, optional CUDAGECKO, MetaGECKO v2 CPU / WGPU / CUDA)
import csv, json, os, re, shutil, subprocess, tempfile, time

def dir_size_mib(path):
    tot = 0
    for root, _, files in os.walk(path):
        for fn in files:
            try:
                tot += os.path.getsize(os.path.join(root, fn))
            except OSError:
                pass
    return tot / (1024 * 1024)

def run_subprocess_measured(cmd, cwd=None):
    with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
        t0 = time.perf_counter()
        p = subprocess.Popen(cmd, cwd=cwd, stdout=out_f, stderr=err_f)
        _, status, ru = os.wait4(p.pid, 0)
        wall_s = time.perf_counter() - t0
        out_f.seek(0); err_f.seek(0)
        stdout = out_f.read().decode('utf-8', errors='replace')
        stderr = err_f.read().decode('utf-8', errors='replace')
        rss_mib = ru.ru_maxrss / 1024.0
    return status, wall_s, rss_mib, stdout, stderr

def run_metagecko_v2(
    backend, tag, read_paths, ref_path,
    wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
    min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL, batch_size='64M',
    multi_map='fractional', formats='abundance,json', keep_outdir=False,
):
    outdir = os.path.join(WORK_DIR, f'v2_{tag}_{backend}')
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    cmd = [
        METAGECKO_V2_BIN,
        *read_paths,
        '--refs', ref_path,
        '-k', str(wl),
        '-l', str(min_len),
        '-s', str(min_sim),
        '--min-read-cov', str(min_cov),
        '--freq-limit', str(fl),
        '--batch-size', str(batch_size),
        '--multi-map', multi_map,
        '--backend', backend,
        '-f', formats,
        '-o', outdir,
    ]
    status, wall_s, rss_mib, stdout, stderr = run_subprocess_measured(cmd)
    if status != 0:
        shutil.rmtree(outdir, ignore_errors=True)
        msg = (stderr.strip() or stdout.strip()).splitlines()
        return {'ok': False, 'error': msg[-1] if msg else f'exit {status}'}, None

    # Parse timing breakdown from MetaGECKO stderr logs
    ref_load_s, ref_index_s = 0.0, 0.0
    m_ref = re.search(r'loaded in ([0-9.]+)s, indexed on \[([^\]]+)\] in ([0-9.]+)s', stderr)
    backend_label = backend
    if m_ref:
        ref_load_s = float(m_ref.group(1))
        backend_label = m_ref.group(2)
        ref_index_s = float(m_ref.group(3))

    sample_logs = []
    for m in re.finditer(
        r"Sample '([^']+)': (\d+)/(\d+) reads mapped \(([0-9.]+)%, unique=(\d+), multi=(\d+)\), (\d+) taxa detected, (\d+) batches \| ([0-9.]+)s \(([0-9]+) reads/s, compute=([0-9.]+)s\)",
        stderr,
    ):
        sample_logs.append({
            'sample': m.group(1),
            'mapped_reads': int(m.group(2)),
            'total_reads': int(m.group(3)),
            'mapped_pct': float(m.group(4)),
            'unique_reads': int(m.group(5)),
            'multi_reads': int(m.group(6)),
            'taxa_detected': int(m.group(7)),
            'batches': int(m.group(8)),
            'sample_wall_s': float(m.group(9)),
            'reads_per_s': int(m.group(10)),
            'compute_s': float(m.group(11)),
        })

    # Load JSON summaries per sample
    sample_jsons = {}
    for fn in sorted(os.listdir(outdir)):
        if fn.endswith('.abundance.json'):
            sname = fn[:-len('.abundance.json')]
            sample_jsons[sname] = json.load(open(os.path.join(outdir, fn)))

    total_reads  = sum(s['total_reads'] for s in sample_logs)
    mapped_reads = sum(s['mapped_reads'] for s in sample_logs)
    total_hsps   = sum(sum(g['total_hsps'] for g in sj['profiles']) for sj in sample_jsons.values())
    compute_s    = sum(s['compute_s'] for s in sample_logs) + ref_index_s

    rec = {
        'ok': True,
        'backend': backend,
        'backend_label': backend_label,
        'wall_s': round(wall_s, 3),
        'ref_load_s': round(ref_load_s, 3),
        'ref_index_s': round(ref_index_s, 3),
        'compute_s': round(compute_s, 3),
        'rss_mib': round(rss_mib, 1),
        'temp_disk_mib': 0.0,
        'total_reads': total_reads,
        'mapped_reads': mapped_reads,
        'mapped_pct': round(100.0 * mapped_reads / max(1, total_reads), 2),
        'total_hsps': total_hsps,
        'throughput_reads_s': round(total_reads / max(1e-6, wall_s), 1),
        'samples': sample_logs,
        'outdir': outdir if keep_outdir else None,
    }
    if not keep_outdir:
        shutil.rmtree(outdir, ignore_errors=True)
    return rec, sample_jsons

def run_legacy_metagecko_c(tag, sample_fastas, genome_fastas, wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM):
    '''Run Original Legacy MetaGECKO (C, otorreno/meta-gecko) following `mgvworkflow.sh`:
       1. `dict <genome.fa> <gdict> 32` for each genome (both strands)
       2. `dict <sample.fa> <mdict> 32 f` for each metagenome (forward strand)
       3. `frag <mdict> <sample.fa> <gdict> <genome.fa> <out> <minS> <minL> <prefix>` for every (sample, genome) pair
       Measures exact wall time, peak RSS, peak temporary disk footprint, and total fragments reported.
    '''
    assert wl % 4 == 0, 'Legacy MetaGECKO requires word length divisible by 4 (prefix = wl/4)'
    prefix = wl // 4
    outdir = os.path.join(WORK_DIR, f'legacy_mg_{tag}')
    shutil.rmtree(outdir, ignore_errors=True)
    dict_dir = os.path.join(outdir, 'dictionaries')
    frag_dir = os.path.join(outdir, 'fragments')
    os.makedirs(dict_dir, exist_ok=True)
    os.makedirs(frag_dir, exist_ok=True)

    dict_bin = os.path.join(LEGACY_MG_DIR, 'dict')
    frag_bin = os.path.join(LEGACY_MG_DIR, 'frag')

    peak_rss = 0.0
    peak_disk = 0.0
    total_frags = 0
    per_genome_frags = {}
    t_wall0 = time.perf_counter()

    try:
        # Step 1: Build reference genome dictionaries (once per genome)
        t_g0 = time.perf_counter()
        g_dicts = {}
        for gfa in genome_fastas:
            gname = os.path.splitext(os.path.basename(gfa))[0]
            gdict = os.path.join(dict_dir, gname)
            st, _, rss, out, err = run_subprocess_measured([dict_bin, gfa, gdict, '32'], cwd=outdir)
            if st != 0:
                raise RuntimeError(f'legacy dict({gname}) failed: {(err or out).strip()[-200:]}')
            peak_rss = max(peak_rss, rss)
            peak_disk = max(peak_disk, dir_size_mib(outdir))
            g_dicts[gname] = (gdict, gfa)
        ref_index_s = time.perf_counter() - t_g0

        # Step 2: Build metagenome dictionaries (forward strand)
        t_m0 = time.perf_counter()
        m_dicts = {}
        for mfa in sample_fastas:
            mname = os.path.splitext(os.path.basename(mfa))[0]
            mdict = os.path.join(dict_dir, mname)
            st, _, rss, out, err = run_subprocess_measured([dict_bin, mfa, mdict, '32', 'f'], cwd=outdir)
            if st != 0:
                raise RuntimeError(f'legacy dict({mname}) failed: {(err or out).strip()[-200:]}')
            peak_rss = max(peak_rss, rss)
            peak_disk = max(peak_disk, dir_size_mib(outdir))
            m_dicts[mname] = (mdict, mfa)
        meta_dict_s = time.perf_counter() - t_m0

        # Step 3: Serial M x G fragment generation (`frag`)
        t_f0 = time.perf_counter()
        for mname, (mdict, mfa) in m_dicts.items():
            per_genome_frags[mname] = {}
            for gname, (gdict, gfa) in g_dicts.items():
                out_prefix = os.path.join(frag_dir, f'{mname}-{gname}')
                # Pass integer string for min_sim (see legacy `is_float` bug in common_functions.c!)
                st, _, rss, out, err = run_subprocess_measured(
                    [frag_bin, mdict, mfa, gdict, gfa, out_prefix, str(int(min_sim)), str(int(min_len)), str(prefix)],
                    cwd=outdir,
                )
                if st != 0:
                    raise RuntimeError(f'legacy frag({mname} x {gname}) failed: {(err or out).strip()[-200:]}')
                peak_rss = max(peak_rss, rss)
                peak_disk = max(peak_disk, dir_size_mib(outdir))
                # Count fragments in legacy .frags file (if produced)
                fpath = out_prefix + '.frags'
                n_fr = 0
                if os.path.exists(fpath):
                    # Legacy MetaGECKO writes ASCII or binary frags; detect header/size safely
                    sz = os.path.getsize(fpath)
                    if sz > 16:
                        with open(fpath, 'rb') as fp:
                            sample_bytes = fp.read(64)
                        if b'\n' in sample_bytes and all(32 <= b < 127 or b in (9, 10, 13) for b in sample_bytes[:32]):
                            n_fr = sum(1 for ln in open(fpath, errors='ignore') if ln.strip() and not ln.startswith('#') and not ln.startswith('Type'))
                        else:
                            n_fr = max(0, (sz - 16) // 93)
                per_genome_frags[mname][gname] = n_fr
                total_frags += n_fr
        frag_s = time.perf_counter() - t_f0
        wall_s = time.perf_counter() - t_wall0

        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend': 'legacy_metagecko',
            'backend_label': 'Legacy MetaGECKO (C 2016, disk-based)',
            'wall_s': round(wall_s, 3),
            'ref_index_s': round(ref_index_s, 3),
            'meta_dict_s': round(meta_dict_s, 3),
            'frag_s': round(frag_s, 3),
            'compute_s': round(wall_s, 3),
            'rss_mib': round(peak_rss, 1),
            'temp_disk_mib': round(peak_disk, 1),
            'total_hsps': total_frags,
        }, per_genome_frags
    except Exception as e:
        shutil.rmtree(outdir, ignore_errors=True)
        return {'ok': False, 'error': str(e)}, None

def run_cudagecko_pairwise(tag, sample_fastas, genome_fastas, min_len=DEFAULT_MIN_LEN):
    '''Optional baseline: Esteban Pérez-Wohlfeil's original CUDAGECKO (pairwise M x G).'''
    if not CUDAGECKO_BIN:
        return {'ok': False, 'error': 'CUDAGECKO not built on Colab CUDA 12'}, None
    outdir = os.path.join(WORK_DIR, f'cudagecko_{tag}')
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    peak_rss = 0.0
    t0 = time.perf_counter()
    try:
        for mfa in sample_fastas:
            for gfa in genome_fastas:
                st, _, rss, out, err = run_subprocess_measured(
                    [CUDAGECKO_BIN, '-query', mfa, '-ref', gfa, '-len', str(min_len), '-dev', '0'],
                    cwd=outdir,
                )
                if st != 0:
                    raise RuntimeError(f'cudagecko failed: {(err or out).strip()[-200:]}')
                peak_rss = max(peak_rss, rss)
        wall_s = time.perf_counter() - t0
        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend': 'cudagecko',
            'backend_label': 'Esteban CUDAGECKO (CUDA pairwise)',
            'wall_s': round(wall_s, 3),
            'compute_s': round(wall_s, 3),
            'rss_mib': round(peak_rss, 1),
            'temp_disk_mib': 0.0,
        }, None
    except Exception as e:
        shutil.rmtree(outdir, ignore_errors=True)
        return {'ok': False, 'error': str(e)}, None

print('Unified MetaGECKO benchmark harness ready.')


In [ ]:
# 5. EXPERIMENT 1: Direct Head-to-Head on Turnbaugh et al. (2009) Lean & Obese Twin Metagenomes
#    Compares:
#      - Original Legacy MetaGECKO (C, otorreno/meta-gecko) on uncompressed FASTA (serial M x G on disk)
#      - Optional Esteban CUDAGECKO (CUDA pairwise)
#      - MetaGECKO v2 Rust AVX2 CPU (prepare-once unified DB, native .fastq.gz streaming)
#      - MetaGECKO v2 Rust WGPU     (persistent GPU-resident DB + WGSL compute shaders)
#      - MetaGECKO v2 Rust CUDA     (persistent VRAM DB + CUB Hash-CSR + warp extension)
import csv, json, os

EXP1_JSON = os.path.join(OUT_DIR, 'exp1_head_to_head.json')
exp1 = json.load(open(EXP1_JSON)) if (RESUME and os.path.exists(EXP1_JSON)) else {}

if not RUN_EXP1_HEAD_TO_HEAD:
    print('Skipping Experiment 1 (RUN_EXP1_HEAD_TO_HEAD = False).')
else:
    sample_fq_gz  = [POOLED_LEAN['fastq_gz'], POOLED_OBESE['fastq_gz']]
    sample_fastas = [POOLED_LEAN['fasta'],    POOLED_OBESE['fasta']]
    genome_fastas = [r['path'] for r in REF_CATALOG]
    total_reads_exp1 = POOLED_LEAN['reads'] + POOLED_OBESE['reads']
    total_bases_exp1 = POOLED_LEAN['bases'] + POOLED_OBESE['bases']
    print(f'Experiment 1 Workload: 2 pooled twin metagenomes ({total_reads_exp1:,} reads, {total_bases_exp1/1e6:.2f} Mbp) x {len(genome_fastas)} gut genomes ({total_ref_bp/1e6:.2f} Mbp)')
    print(f'Parameters: k={DEFAULT_WL}, L={DEFAULT_MIN_LEN}, S={DEFAULT_MIN_SIM}%, min_read_cov={DEFAULT_MIN_COV}%, FL={DEFAULT_FL}\n', flush=True)

    # 1a. MetaGECKO v2 backends (CPU, WGPU, CUDA) — streaming native .fastq.gz directly!
    v2_profiles = {}
    for b in ['cpu', 'wgpu', 'cuda']:
        if b in exp1 and exp1[b].get('ok'):
            print(f'  [metagecko_{b:<4}] already cached: wall={exp1[b]["wall_s"]:.2f}s', flush=True)
            continue
        rec, profs = run_metagecko_v2(
            b, 'exp1_twins', sample_fq_gz, REFS_DIR,
            wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
            min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL,
            keep_outdir=(b == 'cuda'),
        )
        exp1[b] = rec
        if profs:
            v2_profiles[b] = profs
        if rec['ok']:
            print(
                f"  [metagecko_{b:<4}] wall={rec['wall_s']:7.2f}s (ref_idx={rec['ref_index_s']:.2f}s, comp={rec['compute_s']:.2f}s) | "
                f"throughput={rec['throughput_reads_s']:,.0f} reads/s | RSS={rec['rss_mib']:6.1f} MiB | "
                f"mapped={rec['mapped_reads']:,}/{rec['total_reads']:,} ({rec['mapped_pct']:.2f}%) | HSPs={rec['total_hsps']:,}",
                flush=True,
            )
        else:
            print(f"  [metagecko_{b:<4}] ERROR: {rec['error']}", flush=True)
        json.dump(exp1, open(EXP1_JSON, 'w'), indent=2)

    # 1b. Original Legacy MetaGECKO (C, otorreno/meta-gecko) — requires uncompressed FASTA
    if 'legacy_metagecko' not in exp1 or not exp1['legacy_metagecko'].get('ok'):
        print('\n  Running Original Legacy MetaGECKO (C, disk-based serial M x G)...', flush=True)
        rec_leg, leg_frags = run_legacy_metagecko_c(
            'exp1_twins', sample_fastas, genome_fastas,
            wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
        )
        if rec_leg['ok']:
            rec_leg['total_reads'] = total_reads_exp1
            rec_leg['throughput_reads_s'] = round(total_reads_exp1 / max(1e-6, rec_leg['wall_s']), 1)
            print(
                f"  [legacy_c      ] wall={rec_leg['wall_s']:7.2f}s (gdict={rec_leg['ref_index_s']:.2f}s, mdict={rec_leg['meta_dict_s']:.2f}s, frag={rec_leg['frag_s']:.2f}s) | "
                f"throughput={rec_leg['throughput_reads_s']:,.0f} reads/s | RSS={rec_leg['rss_mib']:6.1f} MiB | "
                f"temp_disk={rec_leg['temp_disk_mib']:,.1f} MiB | frags={rec_leg['total_hsps']:,}",
                flush=True,
            )
        else:
            print(f"  [legacy_c      ] ERROR: {rec_leg['error']}", flush=True)
        exp1['legacy_metagecko'] = rec_leg
        json.dump(exp1, open(EXP1_JSON, 'w'), indent=2)
    else:
        print(f'  [legacy_c      ] already cached: wall={exp1["legacy_metagecko"]["wall_s"]:.2f}s')

    # 1c. Optional Esteban CUDAGECKO
    if CUDAGECKO_BIN and ('cudagecko' not in exp1):
        rec_cg, _ = run_cudagecko_pairwise('exp1_twins', sample_fastas, genome_fastas, min_len=DEFAULT_MIN_LEN)
        if rec_cg['ok']:
            rec_cg['total_reads'] = total_reads_exp1
            rec_cg['throughput_reads_s'] = round(total_reads_exp1 / max(1e-6, rec_cg['wall_s']), 1)
            print(f"  [cudagecko     ] wall={rec_cg['wall_s']:7.2f}s | throughput={rec_cg['throughput_reads_s']:,.0f} reads/s | RSS={rec_cg['rss_mib']:6.1f} MiB", flush=True)
        else:
            print(f"  [cudagecko     ] skipped/error: {rec_cg['error']}", flush=True)
        exp1['cudagecko'] = rec_cg
        json.dump(exp1, open(EXP1_JSON, 'w'), indent=2)

    # --- Export Summary Table (CSV, Markdown, LaTeX) ---
    order = ['legacy_metagecko', 'cudagecko', 'cpu', 'wgpu', 'cuda']
    leg_wall = exp1.get('legacy_metagecko', {}).get('wall_s') if exp1.get('legacy_metagecko', {}).get('ok') else None
    cpu_wall = exp1.get('cpu', {}).get('wall_s') if exp1.get('cpu', {}).get('ok') else None

    rows = []
    md = [
        '# Experiment 1: MetaGECKO Head-to-Head on Turnbaugh et al. (2009) Lean & Obese Twins\n',
        f'Workload: **{total_reads_exp1:,} reads ({total_bases_exp1/1e6:.2f} Mbp)** across 2 pooled samples (Lean + Obese) × **{len(genome_fastas)} human gut genomes ({total_ref_bp/1e6:.2f} Mbp)**, `k={DEFAULT_WL}, L={DEFAULT_MIN_LEN}, S={DEFAULT_MIN_SIM}%`.\n',
        '| Implementation | Wall Time (s) | Ref Index (s) | Throughput (reads/s) | Speedup vs Legacy C | Speedup vs Rust CPU | Peak RSS (MiB) | Temp Disk (MiB) |',
        '|---|---:|---:|---:|---:|---:|---:|---:|',
    ]
    tex = [
        r'\begin{table*}[t]', r'\centering',
        r'\caption{Head-to-head comparison of Original Legacy MetaGECKO (C, disk-based) vs. MetaGECKO v2 (Rust CPU, WGPU, and CUDA) on the Turnbaugh et al. (2009) Lean and Obese twin metagenomes against the 20-genome human gut reference panel.}',
        r'\label{tab:metagecko_head_to_head}',
        r'\begin{tabular}{lrrrrrrr}', r'\toprule',
        r'Implementation & Wall (s) & Ref Idx (s) & Reads/s & vs. Legacy C & vs. Rust CPU & RSS (MiB) & Temp Disk (MiB) \\',
        r'\midrule',
    ]
    for k in order:
        r = exp1.get(k, {})
        if not r.get('ok'):
            continue
        sp_leg = (leg_wall / r['wall_s']) if (leg_wall and r['wall_s'] > 0) else None
        sp_cpu = (cpu_wall / r['wall_s']) if (cpu_wall and r['wall_s'] > 0) else None
        lbl = r.get('backend_label', k)
        rows.append({
            'backend': k, 'label': lbl, 'wall_s': r['wall_s'], 'ref_index_s': r.get('ref_index_s', 0.0),
            'throughput_reads_s': r.get('throughput_reads_s', 0.0),
            'speedup_vs_legacy': round(sp_leg, 2) if sp_leg else '',
            'speedup_vs_cpu': round(sp_cpu, 2) if sp_cpu else '',
            'rss_mib': r['rss_mib'], 'temp_disk_mib': r.get('temp_disk_mib', 0.0),
            'mapped_reads': r.get('mapped_reads', ''), 'total_hsps': r.get('total_hsps', ''),
        })
        s_leg = f'**{sp_leg:.1f}×**' if sp_leg else '—'
        s_cpu = f'{sp_cpu:.2f}×' if sp_cpu else '—'
        s_leg_t = f'{sp_leg:.1f}$\\times$' if sp_leg else '--'
        s_cpu_t = f'{sp_cpu:.2f}$\\times$' if sp_cpu else '--'
        md.append(f"| {lbl} | {r['wall_s']:.2f} | {r.get('ref_index_s', 0.0):.2f} | {r.get('throughput_reads_s', 0.0):,.0f} | {s_leg} | {s_cpu} | {r['rss_mib']:.1f} | {r.get('temp_disk_mib', 0.0):,.1f} |")
        tex.append(f"{lbl.replace('_', r'\_')} & {r['wall_s']:.2f} & {r.get('ref_index_s', 0.0):.2f} & {r.get('throughput_reads_s', 0.0):,.0f} & {s_leg_t} & {s_cpu_t} & {r['rss_mib']:.1f} & {r.get('temp_disk_mib', 0.0):,.1f} \\\\")

    tex += [r'\bottomrule', r'\end{tabular}', r'\end{table*}']
    with open(os.path.join(OUT_DIR, 'exp1_head_to_head.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ['backend'])
        w.writeheader(); w.writerows(rows)
    open(os.path.join(OUT_DIR, 'exp1_head_to_head.md'), 'w').write('\n'.join(md) + '\n')
    open(os.path.join(OUT_DIR, 'exp1_head_to_head.tex'), 'w').write('\n'.join(tex) + '\n')
    print('\n' + '\n'.join(md))


In [ ]:
# 6. EXPERIMENT 2: Full Turnbaugh et al. (2009) Paper Cohort Multi-Sample Profiling
#    Streams ALL selected individual SRP000319 WGS twin `.fastq.gz` runs through ONE
#    prepare-once reference session on `cpu`, `wgpu`, and `cuda` — zero re-indexing!
#    Also produces the Lean vs. Obese comparative abundance matrix & Lander-Waterman
#    low-abundance species evidence table.
import csv, json, os, shutil

EXP2_JSON = os.path.join(OUT_DIR, 'exp2_full_cohort.json')
exp2 = json.load(open(EXP2_JSON)) if (RESUME and os.path.exists(EXP2_JSON)) else {}

if not RUN_EXP2_FULL_COHORT:
    print('Skipping Experiment 2 (RUN_EXP2_FULL_COHORT = False).')
else:
    all_sample_paths = [s['fastq_gz'] for s in SAMPLE_CATALOG]
    tot_reads = sum(s['read_count'] for s in SAMPLE_CATALOG)
    tot_bases = sum(s['base_count'] for s in SAMPLE_CATALOG)
    print(f'Experiment 2: Full Cohort — {len(all_sample_paths)} SRP000319 WGS twin samples (~{tot_reads:,} reads, {tot_bases/1e6:.1f} Mbp) x {len(REF_CATALOG)} gut genomes ({total_ref_bp/1e6:.1f} Mbp)\n', flush=True)

    cuda_profiles = None
    for b in ['cpu', 'wgpu', 'cuda']:
        if b in exp2 and exp2[b].get('ok') and b != 'cuda':
            print(f'  [{b:<4}] already cached: wall={exp2[b]["wall_s"]:.2f}s ({exp2[b]["throughput_reads_s"]:,.0f} reads/s)')
            continue
        rec, profs = run_metagecko_v2(
            b, 'exp2_full_cohort', all_sample_paths, REFS_DIR,
            wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
            min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL,
            keep_outdir=(b == 'cuda'),
        )
        exp2[b] = rec
        if b == 'cuda' and rec['ok']:
            cuda_profiles = profs
            # Copy the comparative multi-sample matrices directly to OUT_DIR
            for fn in ('abundance_matrix.rel.tsv', 'abundance_matrix.reads.tsv'):
                src = os.path.join(rec['outdir'], fn)
                if os.path.exists(src):
                    shutil.copy2(src, os.path.join(OUT_DIR, f'exp2_{fn}'))
            # Save full CUDA sample profiles JSON for plotting
            json.dump(profs, open(os.path.join(OUT_DIR, 'exp2_cuda_profiles.json'), 'w'), indent=2)
            shutil.rmtree(rec['outdir'], ignore_errors=True)
            rec['outdir'] = None
        if rec['ok']:
            print(
                f"  [{b:<4}] wall={rec['wall_s']:7.2f}s (ref_idx ONCE={rec['ref_index_s']:.2f}s, comp={rec['compute_s']:.2f}s) | "
                f"throughput={rec['throughput_reads_s']:,.0f} reads/s | RSS={rec['rss_mib']:6.1f} MiB | "
                f"mapped={rec['mapped_reads']:,}/{rec['total_reads']:,} ({rec['mapped_pct']:.2f}%)",
                flush=True,
            )
        else:
            print(f"  [{b:<4}] ERROR: {rec['error']}", flush=True)
        json.dump(exp2, open(EXP2_JSON, 'w'), indent=2)

    # Print Lean vs. Obese Biological Summary + Lander-Waterman Low-Abundance Audit
    prof_path = os.path.join(OUT_DIR, 'exp2_cuda_profiles.json')
    if os.path.exists(prof_path):
        profs = json.load(open(prof_path))
        phylum_by_g = {r['name']: r['phylum'] for r in REF_CATALOG}
        agg = {}
        for sname, sdata in profs.items():
            grp = 'lean' if sname.startswith('lean_') else 'obese'
            for g in sdata['profiles']:
                gn = g['genome_name']
                a = agg.setdefault(gn, {'lean_rel': [], 'obese_rel': [], 'max_breadth': 0.0, 'evenness_vals': [], 'unique_sum': 0, 'multi_sum': 0})
                a[f'{grp}_rel'].append(g['relative_abundance_pct'])
                a['max_breadth'] = max(a['max_breadth'], g['breadth_pct'])
                if g['evenness_ratio'] > 0:
                    a['evenness_vals'].append(g['evenness_ratio'])
                a['unique_sum'] += g['unique_reads']
                a['multi_sum']  += g['multi_reads']

        print('\n=== Lean vs. Obese Twin Cohort Taxonomic Profile & Low-Abundance Evenness (CUDA) ===')
        print(f"{'Genome':<30} {'Phylum':<15} {'Lean %':>8} {'Obese %':>8} {'Unique':>8} {'Multi':>7} {'MaxBrd%':>8} {'Evenness':>8}")
        print('-' * 98)
        bio_rows = []
        for gn, a in sorted(agg.items(), key=lambda kv: -(sum(kv[1]['lean_rel']) + sum(kv[1]['obese_rel']))):
            lm = sum(a['lean_rel']) / max(1, len(a['lean_rel']))
            om = sum(a['obese_rel']) / max(1, len(a['obese_rel']))
            ev = sum(a['evenness_vals']) / max(1, len(a['evenness_vals']))
            ph = phylum_by_g.get(gn, 'Other')
            print(f"{gn:<30} {ph:<15} {lm:8.2f} {om:8.2f} {a['unique_sum']:8,} {a['multi_sum']:7,} {a['max_breadth']:8.2f} {ev:8.3f}")
            bio_rows.append({
                'genome': gn, 'phylum': ph, 'lean_mean_rel_pct': round(lm, 4), 'obese_mean_rel_pct': round(om, 4),
                'unique_reads': a['unique_sum'], 'multi_reads': a['multi_sum'],
                'max_breadth_pct': round(a['max_breadth'], 3), 'mean_evenness_ratio': round(ev, 4),
            })
        with open(os.path.join(OUT_DIR, 'exp2_lean_vs_obese_biology.csv'), 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(bio_rows[0].keys()))
            w.writeheader(); w.writerows(bio_rows)


In [ ]:
# 7. EXPERIMENT 3: Scaling, Word-Length Sensitivity, Streaming Batch-Size, & Multi-Mapping Ablations
import csv, json, os

if not RUN_EXP3_ABLATIONS:
    print('Skipping Experiment 3 (RUN_EXP3_ABLATIONS = False).')
else:
    all_sample_paths = [s['fastq_gz'] for s in SAMPLE_CATALOG]

    # --- 3A: Prepare-Once Reference Amortization across M = 1, 2, 4, 8, N samples ---
    print('=== Ablation 3A: Prepare-Once Reference Amortization (CPU vs. CUDA) ===', flush=True)
    m_counts = sorted(set([m for m in (1, 2, 4, 8, len(all_sample_paths)) if m <= len(all_sample_paths)]))
    amort_rows = []
    for m in m_counts:
        sub = all_sample_paths[:m]
        for b in ['cpu', 'cuda']:
            rec, _ = run_metagecko_v2(
                b, f'ab3a_m{m}', sub, REFS_DIR,
                wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
                min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL,
            )
            if rec['ok']:
                amort_rows.append({
                    'n_samples': m, 'backend': b, 'wall_s': rec['wall_s'],
                    'ref_index_s': rec['ref_index_s'], 'compute_s': rec['compute_s'],
                    'ref_index_share_pct': round(100.0 * rec['ref_index_s'] / max(1e-6, rec['wall_s']), 2),
                    'throughput_reads_s': rec['throughput_reads_s'], 'rss_mib': rec['rss_mib'],
                })
                print(
                    f"  M={m:2d} [{b:<4}] wall={rec['wall_s']:6.2f}s (ref_idx={rec['ref_index_s']:.2f}s = "
                    f"{100.0*rec['ref_index_s']/max(1e-6, rec['wall_s']):4.1f}%) | {rec['throughput_reads_s']:,.0f} reads/s | RSS={rec['rss_mib']:.1f} MiB",
                    flush=True,
                )
    with open(os.path.join(OUT_DIR, 'exp3a_prepare_once_amortization.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(amort_rows[0].keys()) if amort_rows else ['n_samples'])
        w.writeheader(); w.writerows(amort_rows)

    # --- 3B: Seed Word-Length Sensitivity (k = 12, 16, 20) & Bounded-RAM Batch Size (4M, 16M, 64M) ---
    print('\n=== Ablation 3B: Seed Word-Length (k) & Streaming Batch-Size Scaling ===', flush=True)
    sens_rows = []
    pooled_pair = [POOLED_LEAN['fastq_gz'], POOLED_OBESE['fastq_gz']]
    for k in [12, 16, 20]:
        for b in ['cpu', 'cuda']:
            rec, _ = run_metagecko_v2(
                b, f'ab3b_k{k}', pooled_pair, REFS_DIR,
                wl=k, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
                min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL, batch_size='64M',
            )
            if rec['ok']:
                sens_rows.append({
                    'axis': 'word_length', 'param': f'k={k}', 'backend': b,
                    'wall_s': rec['wall_s'], 'compute_s': rec['compute_s'],
                    'rss_mib': rec['rss_mib'], 'mapped_reads': rec['mapped_reads'],
                    'mapped_pct': rec['mapped_pct'], 'throughput_reads_s': rec['throughput_reads_s'],
                })
                print(
                    f"  k={k:2d} [{b:<4}] wall={rec['wall_s']:6.2f}s | {rec['throughput_reads_s']:,.0f} reads/s | "
                    f"mapped={rec['mapped_reads']:,} ({rec['mapped_pct']:.2f}%) | RSS={rec['rss_mib']:.1f} MiB",
                    flush=True,
                )

    for bs in ['4M', '16M', '64M']:
        for b in ['cpu', 'cuda']:
            rec, _ = run_metagecko_v2(
                b, f'ab3b_bs{bs}', pooled_pair, REFS_DIR,
                wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
                min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL, batch_size=bs,
            )
            if rec['ok']:
                sens_rows.append({
                    'axis': 'batch_size', 'param': bs, 'backend': b,
                    'wall_s': rec['wall_s'], 'compute_s': rec['compute_s'],
                    'rss_mib': rec['rss_mib'], 'mapped_reads': rec['mapped_reads'],
                    'mapped_pct': rec['mapped_pct'], 'throughput_reads_s': rec['throughput_reads_s'],
                })
                print(
                    f"  batch={bs:<3} [{b:<4}] wall={rec['wall_s']:6.2f}s | {rec['throughput_reads_s']:,.0f} reads/s | "
                    f"mapped={rec['mapped_reads']:,} | RSS={rec['rss_mib']:.1f} MiB",
                    flush=True,
                )
    with open(os.path.join(OUT_DIR, 'exp3b_sensitivity_and_streaming.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(sens_rows[0].keys()) if sens_rows else ['axis'])
        w.writeheader(); w.writerows(sens_rows)

    # --- 3C: Multi-Mapping Resolution Modes (`fractional` vs. `best` vs. `all`) ---
    print('\n=== Ablation 3C: Multi-Mapping Resolution Modes (CUDA) ===', flush=True)
    mm_rows = []
    for mode in ['fractional', 'best', 'all']:
        rec, profs = run_metagecko_v2(
            'cuda', f'ab3c_{mode}', [POOLED_LEAN['fastq_gz']], REFS_DIR,
            wl=DEFAULT_WL, min_len=DEFAULT_MIN_LEN, min_sim=DEFAULT_MIN_SIM,
            min_cov=DEFAULT_MIN_COV, fl=DEFAULT_FL, multi_map=mode,
        )
        if rec['ok'] and profs:
            s0 = next(iter(profs.values()))
            mm_rows.append({
                'mode': mode, 'wall_s': rec['wall_s'],
                'mapped_reads': s0['mapped_reads'], 'unique_reads': s0['unique_mapped_reads'],
                'multi_reads': s0['multi_mapped_reads'], 'total_hsps': rec['total_hsps'],
            })
            print(
                f"  mode={mode:<10} | wall={rec['wall_s']:5.2f}s | mapped={s0['mapped_reads']:,} "
                f"(unique={s0['unique_mapped_reads']:,}, multi={s0['multi_mapped_reads']:,}) | HSPs={rec['total_hsps']:,}",
                flush=True,
            )
    with open(os.path.join(OUT_DIR, 'exp3c_multimapping_modes.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(mm_rows[0].keys()) if mm_rows else ['mode'])
        w.writeheader(); w.writerows(mm_rows)


In [ ]:
# 8. Generate Publication-Ready 300-DPI Figures (PNG + PDF) & 1-Click ZIP Bundle
import csv, json, os, zipfile
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8.5,
})

# --- Figure 1: 4-Panel Performance & Scaling Summary ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))

# Panel A: Exp 1 — Head-to-Head Wall Time (Legacy C vs. MetaGECKO v2 CPU/WGPU/CUDA)
ax = axes[0, 0]
e1_p = os.path.join(OUT_DIR, 'exp1_head_to_head.json')
if os.path.exists(e1_p):
    e1 = json.load(open(e1_p))
    items = [
        ('legacy_metagecko', 'Legacy MetaGECKO\n(C 2016, disk)', '#95a5a6'),
        ('cudagecko',        'Esteban CUDAGECKO\n(pairwise GPU)', '#e67e22'),
        ('cpu',              'MetaGECKO v2\nRust AVX2 CPU',       '#2980b9'),
        ('wgpu',             'MetaGECKO v2\nWGPU (Vulkan)',       '#8e44ad'),
        ('cuda',             'MetaGECKO v2\nCUDA (T4 GPU)',       '#27ae60'),
    ]
    labels, vals, cols = [], [], []
    for k, lbl, col in items:
        if e1.get(k, {}).get('ok'):
            labels.append(lbl); vals.append(e1[k]['wall_s']); cols.append(col)
    if vals:
        bars = ax.bar(labels, vals, color=cols, edgecolor='black', linewidth=0.6)
        ax.set_yscale('log')
        ax.set_ylabel('Total Wall Time (s, log scale)')
        ax.set_title('(A) Head-to-Head Runtime on Turnbaugh et al. (2009) Twins')
        ax.grid(True, axis='y', which='both', alpha=0.25)
        leg_v = e1.get('legacy_metagecko', {}).get('wall_s') if e1.get('legacy_metagecko', {}).get('ok') else None
        for bar, v in zip(bars, vals):
            txt = f'{v:.2f}s'
            if leg_v and v < leg_v * 0.95:
                txt += f'\n({leg_v/v:.1f}×)'
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.18, txt, ha='center', va='bottom', fontsize=8, fontweight='bold')

# Panel B: Exp 2 — Full Cohort Throughput & Temp Disk (Zero Disk Spill!)
ax = axes[0, 1]
e2_p = os.path.join(OUT_DIR, 'exp2_full_cohort.json')
if os.path.exists(e2_p):
    e2 = json.load(open(e2_p))
    b_list = [('cpu', 'Rust AVX2 CPU', '#2980b9'), ('wgpu', 'WGPU (Vulkan)', '#8e44ad'), ('cuda', 'CUDA (T4)', '#27ae60')]
    lbls, tputs, cols = [], [], []
    for k, lbl, col in b_list:
        if e2.get(k, {}).get('ok'):
            lbls.append(lbl); tputs.append(e2[k]['throughput_reads_s'] / 1000.0); cols.append(col)
    if tputs:
        bars = ax.bar(lbls, tputs, color=cols, edgecolor='black', linewidth=0.6)
        ax.set_ylabel('Sustained Throughput (k reads / s)')
        ax.set_title(f'(B) Full SRP000319 Cohort Throughput ({len(SAMPLE_CATALOG)} Samples × {len(REF_CATALOG)} Genomes)')
        ax.grid(True, axis='y', alpha=0.25)
        for bar, v in zip(bars, tputs):
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.03, f'{v:.1f}k/s', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

# Panel C: Ablation 3A — Prepare-Once Amortization Scaling
ax = axes[1, 0]
ab3a_p = os.path.join(OUT_DIR, 'exp3a_prepare_once_amortization.csv')
if os.path.exists(ab3a_p):
    rows = list(csv.DictReader(open(ab3a_p)))
    for b, lbl, col, mk in [('cpu', 'MetaGECKO v2 CPU', '#2980b9', 'o'), ('cuda', 'MetaGECKO v2 CUDA (T4)', '#27ae60', '*')]:
        sub = sorted([r for r in rows if r['backend'] == b], key=lambda r: int(r['n_samples']))
        if sub:
            xs = [int(r['n_samples']) for r in sub]
            ys = [float(r['wall_s']) for r in sub]
            ax.plot(xs, ys, marker=mk, color=col, linewidth=2.0, markersize=7, label=lbl)
    ax.set_xlabel('Number of Metagenomic Samples Streamed (M)')
    ax.set_ylabel('Total Wall Time (s)')
    ax.set_title('(C) Prepare-Once Reference Amortization Across Samples')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left', frameon=True)

# Panel D: Ablation 3B — Seed Word-Length Sensitivity (k = 12, 16, 20)
ax = axes[1, 1]
ab3b_p = os.path.join(OUT_DIR, 'exp3b_sensitivity_and_streaming.csv')
if os.path.exists(ab3b_p):
    rows = [r for r in csv.DictReader(open(ab3b_p)) if r['axis'] == 'word_length']
    ks = ['k=12', 'k=16', 'k=20']
    xk = np.arange(len(ks))
    w = 0.35
    for i, (b, lbl, col) in enumerate([('cpu', 'CPU', '#2980b9'), ('cuda', 'CUDA (T4)', '#27ae60')]):
        vals = [next((float(r['wall_s']) for r in rows if r['backend'] == b and r['param'] == p), np.nan) for p in ks]
        ax.bar(xk + (i - 0.5)*w, vals, width=w, color=col, edgecolor='black', linewidth=0.5, label=lbl)
    ax.set_xticks(xk); ax.set_xticklabels(ks)
    ax.set_xlabel('Seed Word Length (k)')
    ax.set_ylabel('Wall Time (s)')
    ax.set_title('(D) Seed Sensitivity Frontier (k = 12, 16, 20)')
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend(loc='upper right', frameon=True)

plt.tight_layout()
fig1_png = os.path.join(OUT_DIR, 'metagecko2_paper_figures.png')
fig1_pdf = os.path.join(OUT_DIR, 'metagecko2_paper_figures.pdf')
plt.savefig(fig1_png, dpi=300, bbox_inches='tight')
plt.savefig(fig1_pdf, bbox_inches='tight')
plt.show()

# --- Figure 2: Biological Validation — Lean vs. Obese Gut Microbiome & Lander-Waterman Low-Abundance Evidence ---
bio_csv = os.path.join(OUT_DIR, 'exp2_lean_vs_obese_biology.csv')
if os.path.exists(bio_csv):
    brows = list(csv.DictReader(open(bio_csv)))
    brows = [r for r in brows if float(r['lean_mean_rel_pct']) + float(r['obese_mean_rel_pct']) > 0][:15]
    if brows:
        fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
        names = [r['genome'].replace('_', ' ') for r in brows]
        y = np.arange(len(names))
        h = 0.38
        ax1.barh(y - h/2, [float(r['lean_mean_rel_pct']) for r in brows], height=h, color='#3498db', label='Lean Twins')
        ax1.barh(y + h/2, [float(r['obese_mean_rel_pct']) for r in brows], height=h, color='#e67e22', label='Obese Twins')
        ax1.set_yticks(y); ax1.set_yticklabels(names, fontsize=8)
        ax1.invert_yaxis()
        ax1.set_xlabel('Mean Length-Normalized Relative Abundance (%)')
        ax1.set_title('Lean vs. Obese Twin Microbiome Composition (SRP000319)')
        ax1.grid(True, axis='x', alpha=0.25)
        ax1.legend(loc='lower right')

        # Lander-Waterman breadth vs evenness ratio for low-abundance validation
        brds = [float(r['max_breadth_pct']) for r in brows]
        evs  = [float(r['mean_evenness_ratio']) for r in brows]
        ax2.scatter(brds, evs, s=70, c='#27ae60', edgecolors='black', linewidth=0.6, alpha=0.85)
        ax2.axhline(1.0, color='#7f8c8d', linestyle='--', linewidth=1.0, label='Ideal Lander-Waterman uniform coverage (ratio = 1.0)')
        for r, bx, ey in zip(brows, brds, evs):
            short = r['genome'].split('_')[0][0] + '. ' + '_'.join(r['genome'].split('_')[1:])
            ax2.annotate(short, (bx, ey), fontsize=7.5, xytext=(4, 2), textcoords='offset points')
        ax2.set_xlabel('Max Observed Genome Coverage Breadth (%)')
        ax2.set_ylabel('Lander-Waterman Evenness Ratio (Observed / Expected)')
        ax2.set_title('Low-Abundance Taxon Evidence (Uniform Coverage vs. Pileup)')
        ax2.grid(True, alpha=0.25)
        ax2.legend(loc='lower right', fontsize=8)

        plt.tight_layout()
        fig2_png = os.path.join(OUT_DIR, 'metagecko2_biological_profile.png')
        fig2_pdf = os.path.join(OUT_DIR, 'metagecko2_biological_profile.pdf')
        plt.savefig(fig2_png, dpi=300, bbox_inches='tight')
        plt.savefig(fig2_pdf, bbox_inches='tight')
        plt.show()

# Bundle ALL results into a 1-click download ZIP
zip_path = '/content/metagecko2_paper_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(OUT_DIR)):
        fp = os.path.join(OUT_DIR, fn)
        if os.path.isfile(fp):
            zf.write(fp, arcname=fn)

print(f'\nSaved 300-DPI figures: {fig1_png} & {fig1_pdf}')
print(f'Bundled ALL results ({os.path.getsize(zip_path)/1024:.1f} KiB) into: {zip_path}')
print('Contents:', ', '.join(sorted(os.listdir(OUT_DIR))))
